In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set consistent visual style for all plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# The file is semicolon-separated, not comma-separated - a common quirk of the UCI bank marketing dataset
# FIX: replaced hardcoded local Windows path with a relative path so the notebook
# runs on any machine (the original path only existed on the author's laptop)
df = pd.read_csv("bank-additional.csv", sep=";")

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

# Intial data exploration


In [ ]:
df.info()
df.describe()

In [ ]:
# Categorical columns and their unique values
categorical_cols = df.select_dtypes(include="object").columns.tolist()
categorical_cols.remove("y")  # y is the target, handled separately

print("Categorical feature columns:", categorical_cols)
print()
for col in categorical_cols:
    print(f"{col}: {df[col].unique()}")

In [ ]:
# Target variable distribution - this dataset is imbalanced, which matters for later modelling
print(df["y"].value_counts())
print()
print(df["y"].value_counts(normalize=True).round(3) * 100, "%")

## Data Cleaning

### Duplicate Records

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Number of fully duplicated rows: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicate rows found.")

### Disguised Missing Values ("unknown" category)

In [ ]:
unknown_cols = ["job", "marital", "education", "default", "housing", "loan"]

unknown_summary = pd.DataFrame({
    "unknown_count": [(df[col] == "unknown").sum() for col in unknown_cols],
    "unknown_pct": [round((df[col] == "unknown").mean() * 100, 2) for col in unknown_cols]
}, index=unknown_cols)

unknown_summary

### Sentinel Value in `pdays` (999 = "not previously contacted")


In [ ]:
print("Value counts for pdays = 999 (never previously contacted):")
print((df["pdays"] == 999).sum(), "out of", len(df), "rows")

# Distribution of pdays excluding the sentinel, to see what real values look like
df.loc[df["pdays"] != 999, "pdays"].describe()

##  Feature Engineering

In [ ]:
# Feature engineering: extract a binary flag from the pdays sentinel value
df["was_contacted_before"] = np.where(df["pdays"] == 999, "no", "yes")

print(df["was_contacted_before"].value_counts())
df[["pdays", "was_contacted_before"]].head(10)

In [ ]:
age_bins = [17, 25, 35, 45, 55, 65, 100]
age_labels = ["18-25", "26-35", "36-45", "46-55", "56-65", "66+"]

df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels)

df[["age", "age_group"]].head(10)

## Outlier Check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(y=df["age"], ax=axes[0], color="skyblue")
axes[0].set_title("Age")

sns.boxplot(y=df["duration"], ax=axes[1], color="salmon")
axes[1].set_title("Call Duration (seconds)")

sns.boxplot(y=df["campaign"], ax=axes[2], color="lightgreen")
axes[2].set_title("Number of Contacts (Campaign)")

plt.tight_layout()
plt.show()


## Exploratory Data Analysis (EDA)

### Target Variable Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="y", data=df, palette=["#4C72B0", "#DD8452"])
plt.title("Distribution of Target Variable (Subscribed to Term Deposit)")
plt.xlabel("Subscribed")
plt.ylabel("Count")
plt.show()

print(df["y"].value_counts(normalize=True).round(3) * 100, "%")


### Numeric Feature Distributions


In [ ]:
numeric_cols = ["age", "duration", "campaign", "emp.var.rate",
                "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

df[numeric_cols].hist(bins=20, figsize=(14, 10))
plt.tight_layout()
plt.show()


### Categorical Features vs Target

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), ["job", "marital", "education", "was_contacted_before"]):
    order = df[col].value_counts().index
    sns.countplot(y=col, hue="y", data=df, order=order, ax=ax)
    ax.set_title(f"{col} vs Subscription Outcome")

plt.tight_layout()
plt.show()


In [ ]:
# Age group vs subscription rate
age_group_rate = df.groupby("age_group", observed=True)["y"].apply(
    lambda x: (x == "yes").mean() * 100
).round(2)

plt.figure(figsize=(7, 4))
age_group_rate.plot(kind="bar", color="#4C72B0")
plt.title("Subscription Rate (%) by Age Group")
plt.ylabel("Subscription Rate (%)")
plt.xlabel("Age Group")
plt.xticks(rotation=0)
plt.show()

age_group_rate

### Correlation Between Numeric Features

In [ ]:
plt.figure(figsize=(9, 7))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

### Month and Day-of-Week Distributions


In [ ]:
# Month follows the actual calendar order the campaign ran in (Mar-Dec, no Jan/Feb data)
month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
dow_order = ["mon", "tue", "wed", "thu", "fri"]

present_months = [m for m in month_order if m in df["month"].unique()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x="month", data=df, order=present_months, ax=axes[0], color="#4C72B0")
axes[0].set_title("Number of Contacts by Month")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Count")

sns.countplot(x="day_of_week", data=df, order=dow_order, ax=axes[1], color="#55A868")
axes[1].set_title("Number of Contacts by Day of Week")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
# Subscription RATE (not just raw counts) by month and day - this is the more useful view,
# since it controls for how many calls were made in each period
month_rate = df.groupby("month")["y"].apply(lambda x: (x == "yes").mean() * 100).reindex(present_months)
dow_rate = df.groupby("day_of_week")["y"].apply(lambda x: (x == "yes").mean() * 100).reindex(dow_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

month_rate.plot(kind="bar", ax=axes[0], color="#C44E52")
axes[0].set_title("Subscription Rate (%) by Month")
axes[0].set_ylabel("Subscription Rate (%)")
axes[0].set_xlabel("Month")
axes[0].tick_params(axis="x", rotation=0)

dow_rate.plot(kind="bar", ax=axes[1], color="#8172B2")
axes[1].set_title("Subscription Rate (%) by Day of Week")
axes[1].set_ylabel("Subscription Rate (%)")
axes[1].set_xlabel("Day of Week")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

print("Subscription rate by month (%):")
print(month_rate.round(2))
print()
print("Subscription rate by day of week (%):")
print(dow_rate.round(2))


## Encoding Categorical Variables


In [ ]:
# Target variable: encode "no"/"yes" as 0/1 for classification
df["y"] = df["y"].map({"no": 0, "yes": 1})

# Nominal categorical variables (no natural order) -> one-hot encoding.
# We keep the "unknown" category as its own dummy column rather than imputing it away,
# since "unknown" itself can carry signal (e.g. people who don't disclose default status).
nominal_cols = ["job", "marital", "education", "default", "housing", "loan", "contact",
                "month", "day_of_week", "poutcome", "was_contacted_before"]

df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# age_group IS ordinal (18-25 < 26-35 < ... < 66+), so it gets an ordinal integer
# encoding instead of one-hot, which would throw away the ordering information
age_order = ["18-25", "26-35", "36-45", "46-55", "56-65", "66+"]
df_encoded["age_group"] = df_encoded["age_group"].map({g: i for i, g in enumerate(age_order)})

print(f"Shape after encoding: {df_encoded.shape}")
df_encoded.head()


In [ ]:
# Drop duration - see note above on target leakage
df_encoded = df_encoded.drop(columns=["duration"])
print(f"Final shape used for modelling: {df_encoded.shape}")


## Train/Test Split


In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=["y"])
y = df_encoded["y"]

# stratify=y keeps the same ~89/11 class balance in both the train and test sets,
# which matters given how imbalanced the target is
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print("\nClass balance in training set:")
print(y_train.value_counts(normalize=True).round(3))


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt_model = DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42)
dt_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

print("Both models trained.")


## Model Evaluation


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_auc_score, roc_curve)

def evaluate_model(model, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f}   Precision: {prec:.4f}   Recall: {rec:.4f}   F1: {f1:.4f}   ROC-AUC: {auc:.4f}\n")
    print(classification_report(y_test, y_pred, target_names=["no", "yes"]))

    cm = confusion_matrix(y_test, y_pred)
    return {"name": name, "accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "roc_auc": auc, "confusion_matrix": cm, "y_prob": y_prob}

results = []
results.append(evaluate_model(dt_model, "Decision Tree"))
print()
results.append(evaluate_model(rf_model, "Random Forest"))


In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, res in zip(axes, results):
    sns.heatmap(res["confusion_matrix"], annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["no", "yes"], yticklabels=["no", "yes"])
    ax.set_title(f"{res['name']} - Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
# ROC curves for both models on the same axes
plt.figure(figsize=(7, 6))
for res, color in zip(results, ["#4C72B0", "#DD8452"]):
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    plt.plot(fpr, tpr, label=f"{res['name']} (AUC = {res['roc_auc']:.3f})", color=color, linewidth=2)

plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves: Decision Tree vs Random Forest")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Top 10 most important features for each model
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, model, res in zip(axes, [dt_model, rf_model], results):
    importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
    sns.barplot(x=importances.values, y=importances.index, ax=ax, color="#55A868")
    ax.set_title(f"Top 10 Feature Importances - {res['name']}")
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side metric comparison
metrics_df = pd.DataFrame(results).set_index("name")[["accuracy", "precision", "recall", "f1", "roc_auc"]]
metrics_df.columns = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
print(metrics_df.round(3))

metrics_df.plot(kind="bar", figsize=(10, 6), colormap="viridis")
plt.title("Decision Tree vs Random Forest - Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
